# RUNG-26b DESIGN (torch) — RFdiffusion + ProteinMPNN -> binder sequences

GPU-GUARDED + live heartbeat (after the 3.4h-on-CPU disaster). Generates binder sequences, saves to Drive,
NO jax (scoring is the separate `binder_score_colab`). Run: Cell1 -> 2 (install) -> 4 (batch). The GPU assert
fails LOUD if you're on a CPU runtime, and Cell 4 prints a heartbeat with the live backbone count.

In [ ]:
#@title Cell 1 — clone + GPU CHECK + load folded target
import os, json
from pathlib import Path
REPO=Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), '❌ NO GPU — Runtime > Change runtime type > T4 GPU, then re-run. Do NOT proceed on CPU (RFdiffusion is ~50x slower).'
print('GPU:', torch.cuda.get_device_name(0))
from google.colab import drive; drive.mount('/content/drive')
M=json.load(open('/content/drive/MyDrive/cancer-recon/rung26b_rfdiff/meta.json'))
print('target:', M['target_id'], '| hotspot B'+str(M['hotspot']))
print('[CELL 1] OK')

In [ ]:
#@title Cell 2 — install RFdiffusion + ProteinMPNN (torch 2.4 family; NO jax)  [~12 min]
import torch
assert torch.cuda.is_available(), '❌ NO GPU — switch to T4 first.'
import os, glob, subprocess
def sh(c): r=subprocess.run(c,shell=True,capture_output=True,text=True); (r.returncode and print('  !',c[:60],'\n',(r.stderr or '')[-400:])); return r.returncode
sh('pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu124')
sh('pip install -q gemmi')
if not os.path.exists('/content/RFdiffusion'):
    sh('apt-get -qq install -y aria2 >/dev/null')
    sh('git clone -q https://github.com/sokrypton/RFdiffusion.git /content/RFdiffusion')
    sh('git clone -q https://github.com/dauparas/ProteinMPNN.git /content/ProteinMPNN')
    sh('pip install -q jedi omegaconf hydra-core icecream pyrsistent pynvml decorator')
    sh('pip install -q git+https://github.com/NVIDIA/dllogger#egg=dllogger')
    sh('pip install -q --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html')
    sh('pip install -q --no-dependencies e3nn==0.5.5 opt_einsum_fx')
    sh('cd /content/RFdiffusion/env/SE3Transformer && pip install -q --no-dependencies .')
    os.makedirs('/content/RFdiffusion/models', exist_ok=True)
    sh('cd /content/RFdiffusion/models && aria2c -q -x16 http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt')
sh('pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu124')
import torch as _t; print('torch', _t.__version__, '| cuda', _t.cuda.is_available())
RI=glob.glob('/content/RFdiffusion/scripts/run_inference.py')+glob.glob('/content/RFdiffusion/run_inference.py')
print('run_inference.py:', RI[0] if RI else 'NOT FOUND', '| ckpt:', bool(glob.glob('/content/RFdiffusion/models/Complex_base_ckpt.pt')))
print('[CELL 2] done')

In [ ]:
#@title Cell 3 — SMOKE: 1 binder sequence (RFdiffusion -> ProteinMPNN)  [~3-5 min]
import os, glob, subprocess, json, gemmi
from google.colab import drive; drive.mount('/content/drive')
M=json.load(open('/content/drive/MyDrive/cancer-recon/rung26b_rfdiff/meta.json'))
WORK=M['work']; MUT_PDB=M['mut_pdb']; HOT=f"B{M['hotspot']}"; GL=M['groove_len']; PL=M['pep_len']
RI=(glob.glob('/content/RFdiffusion/scripts/run_inference.py')+glob.glob('/content/RFdiffusion/run_inference.py'))[0]
CONTIG=f"[A1-{GL}/0 B1-{PL}/0 70-90]"
def binder_chain(pdb):
    st=gemmi.read_structure(pdb)
    for c in st[0]:
        n=len(c)
        if n!=GL and n!=PL and 50<=n<=120: return c.name
    return [c.name for c in st[0]][-1]
def rfdiffuse(prefix,n):
    cmd=(f"cd /content/RFdiffusion && python {RI} inference.input_pdb={MUT_PDB} 'contigmap.contigs={CONTIG}' "
         f"'ppi.hotspot_res=[{HOT}]' inference.output_prefix={prefix} inference.num_designs={n} inference.ckpt_override_path=models/Complex_base_ckpt.pt")
    return subprocess.run(cmd,shell=True)   # STREAM live (RFdiffusion prints 'Making design N' per design)
def mpnn(bb, outdir=None):
    ch=binder_chain(bb); out=(outdir or os.path.dirname(bb))+'/mpnn'; os.makedirs(out,exist_ok=True)
    sd=abs(hash(bb))%100000
    subprocess.run(f"cd /content/ProteinMPNN && python protein_mpnn_run.py --pdb_path {bb} --pdb_path_chains {ch} --out_folder {out} --num_seq_per_target 1 --sampling_temp 0.1 --seed {sd} --batch_size 1",shell=True,capture_output=True,text=True)
    fa=sorted(glob.glob(f'{out}/seqs/*.fa'))
    if not fa: return None
    s=[l.strip() for l in open(fa[-1]) if l.strip() and not l.startswith('>')][-1]
    return s.split('/')[0] if '/' in s else s
rfdiffuse(f'{WORK}/smoke/d',1); bb=sorted(glob.glob(f'{WORK}/smoke/*.pdb'))
print('backbone:', bb[-1] if bb else 'NONE')
if bb:
    seq=mpnn(bb[-1]); print('binder seq:', seq, '| len', len(seq) if seq else 0)
    print('[CELL 3] SMOKE OK — launch Cell 4' if seq else '[CELL 3] MPNN failed')

In [ ]:
#@title Cell 4 — BATCH: N DIVERSE designs + live HEARTBEAT (resumable)  [GPU]
N_DESIGNS=50 #@param {type:'integer'}
import torch
assert torch.cuda.is_available(), '❌ NO GPU — switch to T4. (RFdiffusion on CPU = ~50x slower, never finishes.)'
import os, glob, json, time, threading
from google.colab import drive; drive.mount('/content/drive')
M=json.load(open('/content/drive/MyDrive/cancer-recon/rung26b_rfdiff/meta.json')); WORK=M['work']
BATCH=f'{WORK}/batch'; DES=f'{WORK}/designs'; os.makedirs(BATCH,exist_ok=True); os.makedirs(DES,exist_ok=True)
# LIVE HEARTBEAT: prints backbone count + GPU mem every 30s so you SEE progress during the long RFdiffusion call
_stop=threading.Event()
def _hb():
    t0=time.time()
    while not _stop.is_set():
        n=len(glob.glob(f'{BATCH}/d_*.pdb'))
        print(f"[hb {time.strftime('%H:%M:%S')}] backbones {n}/{N_DESIGNS} | {int(time.time()-t0)}s | gpu {torch.cuda.memory_allocated()/1e9:.1f}GB", flush=True)
        _stop.wait(30)
threading.Thread(target=_hb, daemon=True).start()
print(f'RFdiffusion: {N_DESIGNS} diverse backbones (resumable; streams live below + heartbeat)...')
rfdiffuse(f'{BATCH}/d', N_DESIGNS)
_stop.set(); time.sleep(0.5)
bbs=sorted(glob.glob(f'{BATCH}/d_*.pdb')); print(f'\n{len(bbs)} backbones; ProteinMPNN per backbone...')
for bb in bbs:
    idx=int(os.path.basename(bb).replace('d_','').replace('.pdb',''))
    did=f'design_{idx:04d}'; dd=f'{DES}/{did}'; os.makedirs(dd,exist_ok=True)
    if os.path.exists(f'{dd}/seq.json'): continue
    try:
        seq=mpnn(bb, dd)
        if not seq: raise RuntimeError('no seq')
        json.dump({'design_id':did,'target':M['target_id'],'sequence':seq}, open(f'{dd}/seq.json','w'))
        print(f'  {did}: len {len(seq)}', flush=True)
    except Exception as e: print(f'  {did} FAIL: {e}', flush=True)
allseq=[json.load(open(f)).get('sequence') for f in glob.glob(f'{DES}/*/seq.json')]
print(f'[CELL 4] done — {len(allseq)} sequences, {len(set(s for s in allseq if s))} UNIQUE. Now run binder_score_colab.')